In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.transformer_lstm_svm import Transformer_LSTM_SVM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "Transformer_LSTM_SVM"


device: NVIDIA A30


In [ ]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-3,
                 seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    # two-stage model
    model = Transformer_LSTM_SVM(n_classes=8, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")
        print(f"  backbone params: {n_params:,}")
        print("  --- Stage 1: training Transformer-LSTM backbone ---")

    # stage 1: train transformer-LSTM with softmax head
    with Timer(device) as stage1_timer:
        model.fit_backbone(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr,
                       batch_size=50, seed=seed, verbose=verbose)
                       # remove weight_decay arg

    if verbose:
        print("  --- Stage 2: fitting RBF-SVM on FC(8) logits ---")

    # stage 2: fit SVM on backbone's FC(8) logits (CPU-bound)
    with Timer(device=None) as stage2_timer:
        model.fit_svm(X_tr, y_tr)

    peak_mem_mb = get_gpu_peak_memory_mb(device)
    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed

    # inference timing
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # test evaluation
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(backbone)={stage1_timer.elapsed:.1f}s  "
              f"stage2(SVM)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario":  scenario_idx,
        "model":     MODEL_NAME,
        "n_train":   len(X_tr),
        "accuracy":  acc,
        "precision": p,
        "recall":    r,
        "f1":        f,
        "confusion": cm,
        "n_params":  n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }

def run_scenario_multi_seed(scenario_idx, scenario_dir, device,
                            epochs=200, lr=1e-3, seeds=range(10),
                            verbose=True):
    """Run a scenario across multiple seeds, return mean ± std."""
    seed_runs = []
    for s in seeds:
        if verbose:
            print(f"\n--- Scenario {scenario_idx}, seed={s} ---")
        r = run_scenario(scenario_idx, scenario_dir, device,
                         epochs=epochs, lr=lr, seed=s,
                         verbose=False)
        seed_runs.append(r)
        if verbose:
            print(f"  seed {s}: acc={r['accuracy']:.4f}  F1={r['f1']:.4f}")

    metric_keys = ["accuracy", "precision", "recall", "f1"]
    agg = {
        "scenario":  scenario_idx,
        "model":     seed_runs[0]["model"],
        "n_train":   seed_runs[0]["n_train"],
        "n_params":  seed_runs[0]["n_params"],
        "n_seeds":   len(list(seeds)),
    }
    for k in metric_keys:
        vals = np.array([r[k] for r in seed_runs])
        agg[k] = vals.mean()
        agg[f"{k}_std"] = vals.std(ddof=1)

    agg["train_sec"] = np.mean([r["train_sec"] for r in seed_runs])
    agg["train_sec_stage1"] = np.mean([r["train_sec_stage1"] for r in seed_runs])
    agg["train_sec_stage2"] = np.mean([r["train_sec_stage2"] for r in seed_runs])
    agg["inf_ms_per_sample"] = np.mean([r["inf_ms_per_sample"] for r in seed_runs])
    agg["peak_mem_mb"] = np.mean([r["peak_mem_mb"] for r in seed_runs])
    
    # Keep individual seed results for variance analysis
    agg["per_seed_f1"] = [r["f1"] for r in seed_runs]
    agg["per_seed_acc"] = [r["accuracy"] for r in seed_runs]

    if verbose:
        print(f"\n  === Scenario {scenario_idx} aggregated over {len(list(seeds))} seeds ===")
        print(f"  F1: {agg['f1']:.4f} ± {agg['f1_std']:.4f}")
        print(f"  Acc: {agg['accuracy']:.4f} ± {agg['accuracy_std']:.4f}")

    return agg

In [ ]:
SEEDS = list(range(10))   # 0 through 9

results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario_multi_seed(i, sc_dir, device,
                                epochs=200, lr=1e-3, seeds=SEEDS)
    results.append(r)

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



=== Scenario 1 (Transformer_LSTM_SVM) ===
  shapes: train=(7858, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train labels: [ 858 1000 1000 1000 1000 1000 1000 1000]
  backbone params: 10,869
  --- Stage 1: training Transformer-LSTM backbone ---
  [stage1] ep   1 | loss 2.0287 acc 0.163 | val loss 1.9479 acc 0.186
  [stage1] ep  20 | loss 1.2348 acc 0.491 | val loss 1.2329 acc 0.487
  [stage1] ep  40 | loss 0.4820 acc 0.769 | val loss 0.4082 acc 0.793
  [stage1] ep  60 | loss 0.3576 acc 0.822 | val loss 0.3479 acc 0.818
  [stage1] ep  80 | loss 0.3032 acc 0.854 | val loss 0.3213 acc 0.852
  [stage1] ep 100 | loss 0.2034 acc 0.919 | val loss 0.2336 acc 0.904
  [stage1] ep 120 | loss 0.1472 acc 0.944 | val loss 0.1928 acc 0.937
  [stage1] ep 140 | loss 0.1174 acc 0.956 | val loss 0.1530 acc 0.954
  [stage1] ep 160 | loss 0.0937 acc 0.967 | val loss 0.1556 acc 0.950
  [stage1] ep 180 | loss 0.0792 acc 0.971 | val loss 0.1181 acc 0.969
  [stage1] ep 200 | loss 0.0598 acc 0.980 | val loss

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep   1 | loss 2.0792 acc 0.139 | val loss 2.0565 acc 0.154
  [stage1] ep  20 | loss 1.2597 acc 0.481 | val loss 1.2697 acc 0.489
  [stage1] ep  40 | loss 1.2388 acc 0.487 | val loss 1.2493 acc 0.488
  [stage1] ep  60 | loss 1.0554 acc 0.556 | val loss 1.0446 acc 0.574
  [stage1] ep  80 | loss 0.4249 acc 0.792 | val loss 0.5322 acc 0.738
  [stage1] ep 100 | loss 0.4071 acc 0.798 | val loss 0.6140 acc 0.724
  [stage1] ep 120 | loss 0.3717 acc 0.809 | val loss 0.5598 acc 0.738
  [stage1] ep 140 | loss 0.3732 acc 0.811 | val loss 0.5326 acc 0.750
  [stage1] ep 160 | loss 0.3534 acc 0.822 | val loss 0.5937 acc 0.738
  [stage1] ep 180 | loss 0.3323 acc 0.837 | val loss 0.5176 acc 0.769
  [stage1] ep 200 | loss 0.3291 acc 0.843 | val loss 0.5602 acc 0.770
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.8725  P=0.8756  R=0.8725  F1=0.8722
  COMPUTE: stage1(backbone)=158.0s  stage2(SVM)=0.1s  total=158.1s  inf=0.079ms/sample

=== Scenario 3 (Transformer_LSTM_SVM) ===

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


...

  [stage1] ep  20 | loss 1.9820 acc 0.148 | val loss 1.9572 acc 0.187
  [stage1] ep  40 | loss 1.8972 acc 0.251 | val loss 1.9039 acc 0.292
  [stage1] ep  60 | loss 1.5020 acc 0.376 | val loss 1.4892 acc 0.415
  [stage1] ep  80 | loss 1.3134 acc 0.494 | val loss 1.3403 acc 0.442
  [stage1] ep 100 | loss 1.2692 acc 0.481 | val loss 1.3075 acc 0.459
  [stage1] ep 120 | loss 1.2582 acc 0.488 | val loss 1.3465 acc 0.444
  [stage1] ep 140 | loss 1.2201 acc 0.501 | val loss 1.3752 acc 0.439
  [stage1] ep 160 | loss 1.2106 acc 0.499 | val loss 1.3233 acc 0.454
  [stage1] ep 180 | loss 1.1807 acc 0.514 | val loss 1.3611 acc 0.442
  [stage1] ep 200 | loss 1.1872 acc 0.519 | val loss 1.3461 acc 0.451
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.4925  P=0.5636  R=0.4925  F1=0.4960
  COMPUTE: stage1(backbone)=12.8s  stage2(SVM)=0.0s  total=12.8s  inf=0.020ms/sample

=== Scenario 5 (Transformer_LSTM_SVM) ===
  shapes: train=(238, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  tra

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep  20 | loss 2.0688 acc 0.130 | val loss 2.0409 acc 0.204
  [stage1] ep  40 | loss 1.9178 acc 0.189 | val loss 1.9538 acc 0.192
  [stage1] ep  60 | loss 1.8096 acc 0.349 | val loss 1.8470 acc 0.271
  [stage1] ep  80 | loss 1.5296 acc 0.387 | val loss 1.5602 acc 0.351
  [stage1] ep 100 | loss 1.3718 acc 0.437 | val loss 1.3814 acc 0.451
  [stage1] ep 120 | loss 1.2579 acc 0.492 | val loss 1.3746 acc 0.446
  [stage1] ep 140 | loss 1.2193 acc 0.563 | val loss 1.3376 acc 0.458
  [stage1] ep 160 | loss 1.1874 acc 0.529 | val loss 1.3080 acc 0.462
  [stage1] ep 180 | loss 1.2054 acc 0.529 | val loss 1.2808 acc 0.476
  [stage1] ep 200 | loss 1.1194 acc 0.555 | val loss 1.2830 acc 0.492
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.5162  P=0.6365  R=0.5163  F1=0.5303
  COMPUTE: stage1(backbone)=9.6s  stage2(SVM)=0.0s  total=9.6s  inf=0.016ms/sample

=== Scenario 6 (Transformer_LSTM_SVM) ===
  shapes: train=(77, 1, 13) val=(1600, 1, 13) test=(1600, 1, 13)
  train 

/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  [stage1] ep  20 | loss 2.0567 acc 0.169 | val loss 2.0775 acc 0.154
  [stage1] ep  40 | loss 1.9853 acc 0.260 | val loss 2.0433 acc 0.197
  [stage1] ep  60 | loss 1.9015 acc 0.312 | val loss 2.0235 acc 0.214
  [stage1] ep  80 | loss 1.7792 acc 0.338 | val loss 2.0343 acc 0.237
  [stage1] ep 100 | loss 1.7140 acc 0.351 | val loss 1.9613 acc 0.266
  [stage1] ep 120 | loss 1.5681 acc 0.390 | val loss 1.9053 acc 0.289
  [stage1] ep 140 | loss 1.5488 acc 0.403 | val loss 1.7687 acc 0.326
  [stage1] ep 160 | loss 1.4144 acc 0.442 | val loss 1.6490 acc 0.356
  [stage1] ep 180 | loss 1.3521 acc 0.455 | val loss 1.6555 acc 0.347
  [stage1] ep 200 | loss 1.2498 acc 0.532 | val loss 1.6429 acc 0.375
  --- Stage 2: fitting RBF-SVM on FC(8) logits ---
  TEST: acc=0.3613  P=0.3386  R=0.3612  F1=0.3330
  COMPUTE: stage1(backbone)=3.2s  stage2(SVM)=0.0s  total=3.2s  inf=0.006ms/sample


In [ ]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "accuracy_std",
                       "precision", "precision_std",
                       "recall", "recall_std",
                       "f1", "f1_std",
                       "n_params", "train_sec",
                       "train_sec_stage1", "train_sec_stage2",
                       "inf_ms_per_sample", "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "accuracy_std",
              "precision", "precision_std",
              "recall", "recall_std",
              "f1", "f1_std"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios (mean over {len(SEEDS)} seeds) ===")
print(summary.to_string(index=False))
summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)

# Also save per-seed details
per_seed = pd.DataFrame([
    {"scenario": r["scenario"], "seed": s, "f1": f1, "acc": acc}
    for r in results
    for s, f1, acc in zip(SEEDS, r["per_seed_f1"], r["per_seed_acc"])
])
per_seed.to_csv(f"{MODEL_NAME.lower()}_per_seed.csv", index=False)


=== Transformer_LSTM_SVM summary across scenarios ===
 scenario                model  n_train  accuracy  precision  recall     f1  n_params  train_sec  train_sec_stage1  train_sec_stage2  inf_ms_per_sample  peak_mem_mb
        1 Transformer_LSTM_SVM     7858    0.9594     0.9594  0.9594 0.9593     10869     334.62            334.43              0.19             0.0456        325.5
        2 Transformer_LSTM_SVM     3939    0.8725     0.8756  0.8725 0.8722     10869     158.09            157.98              0.11             0.0792        175.3
        3 Transformer_LSTM_SVM      786    0.4894     0.5798  0.4894 0.4795     10869      32.81             32.77              0.03             0.0400         86.1
        4 Transformer_LSTM_SVM      391    0.4925     0.5636  0.4925 0.4960     10869      12.81             12.80              0.01             0.0200         86.1
        5 Transformer_LSTM_SVM      238    0.5162     0.6365  0.5163 0.5303     10869       9.58              9.58      

In [ ]:
# for r in results:
#     print(f"\nScenario {r['scenario']}  ({r['model']}, "
#           f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
#     print(r["confusion"])


Scenario 1  (Transformer_LSTM_SVM, n_train=7858, acc=0.959)
[[184   1   0  12   0   0   3   0]
 [  0 198   0   1   0   0   0   1]
 [  0   0 196   0   0   4   0   0]
 [ 13   0   0 187   0   0   0   0]
 [  0   0   0   0 200   0   0   0]
 [  0   1   2   0   0 184   0  13]
 [  0   0   0   2   0   0 198   0]
 [  0   7   1   0   0   4   0 188]]

Scenario 2  (Transformer_LSTM_SVM, n_train=3939, acc=0.873)
[[138   3   0  58   0   0   0   1]
 [  1 185   0   0   0   1   0  13]
 [  0   0 198   0   0   2   0   0]
 [ 40   0   0 156   0   0   4   0]
 [  0   0   0   0 200   0   0   0]
 [  0   1   4   0   0 147   0  48]
 [  0   0   0   2   0   0 198   0]
 [  0   8   0   0   0  18   0 174]]

Scenario 3  (Transformer_LSTM_SVM, n_train=786, acc=0.489)
[[ 13   0  63  75   0   1  22  26]
 [  6  79  62   0   2   1  23  27]
 [  0   0 188   1   0   1   0  10]
 [ 18   1  50  70   2   0  32  27]
 [  0   0   0   0 199   1   0   0]
 [  0   0  81   4   0  91   2  22]
 [ 12   0  30  33   0   0  99  26]
 [  0   0 1